# LLMs — Arquitectura, Ciclo de Vida y Prompting

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*De qué está hecho un LLM, cómo aprende, cómo se le habla.*


## Recap Clase 1 → de qué partimos

```
  Texto      ──▶  Tokenización  ──▶  Embeddings  ──▶  LLM  ──▶  Output
  (crudo)          (subpalabras)      (vectores)        ↑hoy
```

| Lo que ya tenemos de Clase 1 | Lo que abrimos hoy |
|---|---|
| Tokenizers (BPE, WordPiece) | Esos mismos en GPT, Llama, Qwen |
| Embeddings de 384 dimensiones | Cómo el Transformer los procesa |
| BoW / TF-IDF como baseline | Por qué attention los supera |


## Hoy hablamos con un LLM **por código**, no por UI

- ChatGPT, Claude, Gemini son apps. Lo que está abajo es un modelo.
- En este curso vamos a tratarlo como una **API** que se invoca desde Python.
- Eso permite: scripting, integración, control fino, prompts reproducibles.

> Cuando deployás IA en una empresa, casi nunca usás la UI. Usás el modelo desde código.


## Demo: hola Groq desde un notebook

Vamos a abrir un Colab y conectarnos a Groq con tres consultas progresivas:

1. **Hola simple** — confirmar que la conexión funciona.
2. **Mismo prompt con `temperature=0` y `temperature=1`** — ver qué cambia.
3. **System prompt "respondé como un pirata"** — introducir roles.

📓 **Notebook**: [`clase02/notebooks/01_groq_intro.ipynb`](notebooks/01_groq_intro.ipynb)
   Tiene un botón "Open in Colab" arriba. Abrilo y seguilo en paralelo.

> No vamos a leer el código en clase, lo ejecutamos.


## Recorrido de la clase

<img src="figures/roadmap.svg" alt="Roadmap de los 7 bloques de la clase, con dos cortes intermedios" class="slide-figure" style="width: 92%;"/>

Vamos primero a entender **qué es** un LLM (bloques 1–4), después a **usarlo bien** (bloque 5), y finalmente a **operarlo en producción** (bloque 6). Cerramos conectando con Clase 3.


# Bloque 1
## ¿Qué es un LLM?

---

*Karpathy, "Intro to LLMs" — la intuición más limpia que existe.*


## Un LLM, en disco, son dos archivos

<img src="figures/two_files.svg" alt="Un LLM = parameters (140GB) + run.c (~500 líneas), corriendo offline en una MacBook" class="slide-figure" style="width: 88%;"/>

Para correr un modelo open weight como **Llama 2 70B** alcanza con un blob de **parámetros** (~140 GB) y un programa que sabe **multiplicar matrices**. Toda la inteligencia vive en los 140 GB.


## Inferencia: 100% local, sin internet

<img src="figures/inference_macbook.svg" alt="Una MacBook ejecutando un LLM offline, sin WiFi y sin API key" class="slide-figure" style="width: 88%;"/>

Una vez que tenés los pesos, el modelo es tuyo. No hay servicio externo en el medio. Esto es lo que cambia con los modelos open weight.


## Entrenar = comprimir un trozo de internet

<img src="figures/training_compression.svg" alt="10 TB de texto → cluster de 6.000 GPUs durante 12 días → 140 GB de pesos" class="slide-figure" style="width: 88%;"/>

Los 140 GB no salen de la nada: son una **compresión con pérdida** de billones de tokens del scraping de internet. Ratio aproximado: **70:1**.

> 📖 *Touvron, H., et al. (2023). Llama 2: Open Foundation and Fine-Tuned Chat Models.*


## La red predice la próxima palabra

<img src="figures/next_word_prediction.svg" alt="Una red neuronal recibe el contexto 'cat sat on a' y predice 'mat' con 97% de probabilidad" class="slide-figure" style="width: 88%;"/>

El objetivo de entrenamiento es **brutalmente simple**: dado un contexto, ¿qué token sigue?

Pero ese objetivo, repetido sobre billones de tokens, **fuerza** a la red a aprender un montón de cosas sobre el mundo.


## Predecir tokens fuerza a aprender el mundo

<img src="figures/world_knowledge.svg" alt="Un artículo de Wikipedia con hechos clave subrayados: fechas, lugares, personas, relaciones" class="slide-figure" style="width: 76%;"/>

Para predecir cada palabra subrayada, la red tiene que **saber** un hecho del mundo: fechas, lugares, relaciones de parentesco, causalidad histórica, gramática, formato Wikipedia.


## La red alucina documentos

<img src="figures/network_dreams.svg" alt="Tres documentos inventados por la red: código Java, ficha de Amazon, artículo de Wikipedia" class="slide-figure" style="width: 88%;"/>

Si dejás al modelo generar libre, **inventa** documentos plausibles: código que compila, fichas de productos con ISBN coherentes, artículos de Wikipedia bien estructurados.

**Ninguno existe.** Esto es la **alucinación**: por arquitectura, el modelo siempre genera el documento más plausible que puede, exista o no en la realidad.


## ¿Cómo funciona por dentro? Casi nadie lo sabe

<img src="figures/inscrutable.svg" alt="Diagrama Transformer simplificado a la izquierda; ejemplo del 'reversal curse' Obama/Stanley Ann Dunham a la derecha" class="slide-figure" style="width: 88%;"/>

Sabemos **ajustar** los parámetros para que prediga mejor. No sabemos cómo **colaboran** los miles de millones de parámetros para hacerlo.

> 📖 *Berglund, L., et al. (2023). The Reversal Curse: LLMs trained on "A is B" fail to learn "B is A".*


# Bloque 2
## Transformer a grandes rasgos

---

*Lo que hay adentro de la red, sin entrar en la matemática.*


## De RNN secuencial a Transformer en paralelo

<img src="figures/rnn_vs_transformer.svg" alt="Comparativa: RNN procesa una palabra a la vez en secuencia; Transformer mira todas a la vez con atención global" class="slide-figure" style="width: 88%;"/>

> 📖 *Vaswani, A., et al. (2017). Attention Is All You Need. NeurIPS 2017. https://arxiv.org/abs/1706.03762*


## La arquitectura Transformer original

<img src="figures/transformer_architecture.svg" alt="Arquitectura completa del Transformer: encoder a la izquierda, decoder a la derecha, conectados por cross-attention" class="slide-figure" style="width: 80%;"/>

Vaswani et al. (2017) la propusieron para **traducción**. La idea: el encoder "lee y entiende" la entrada, el decoder genera la salida token a token, y el cross-attention los conecta.

> A partir de acá la familia se dividió: la mayoría de los LLMs modernos usan **solo el decoder** — lo vemos en el próximo slide.


## Encoder vs Decoder — las dos mitades por separado

<img src="figures/encoder_decoder.svg" alt="Encoder con atención bidireccional (BERT) vs Decoder con atención causal (GPT, Llama, Claude)" class="slide-figure" style="width: 88%;"/>

> En este curso usamos modelos **decoder-only**. Son los que generan respuestas token a token.


## ¿Qué significa que un token "mira" a otro?

<img src="figures/token_mira.svg" alt="El vector de 'banco' se actualiza absorbiendo información de 'quebró', desambiguando su sentido" class="slide-figure" style="width: 88%;"/>

Cada token entra al Transformer como un **vector** de números. "Mirar" es una metáfora: el vector del token se **actualiza absorbiendo información** de los otros tokens, ponderada por un **peso de atención**.

Los pesos los calcula el mecanismo **Query / Key / Value** — próximo slide.


## Attention resuelve la polisemia por contexto

<img src="figures/attention_polysemy.svg" alt="La palabra 'banco' adopta significados distintos según con qué otras palabras se conecte por atención" class="slide-figure" style="width: 88%;"/>

La misma palabra puede significar cosas distintas. La atención **decide** a quién prestarle atención y eso desambigua el sentido.


## Query, Key, Value — la metáfora del buscador

<img src="figures/qkv.svg" alt="Tres cajas Q/K/V con la metáfora 'qué busco / qué tengo / qué devuelvo'" class="slide-figure" style="width: 88%;"/>

Para cada token, el modelo genera tres vectores. **Match(Q, K) → pesos · V** te da una nueva representación que absorbe contexto.

Es un buscador interno, hecho por la red, que se entrena solo.


## ¿Dónde vive Q / K / V dentro del Transformer?

<img src="figures/qkv_in_transformer.svg" alt="Los tres bloques de atención: encoder self-attention, decoder masked self-attention, decoder cross-attention. Q/K/V aparece en los tres; cambia de dónde sale cada vector" class="slide-figure" style="width: 80%;"/>

El mecanismo Q/K/V es **siempre el mismo**. Lo que cambia es de dónde sale cada vector:

- **Encoder self-attention** y **decoder masked self-attention** → Q, K, V salen del mismo input.
- **Cross-attention** (solo en encoder-decoder) → Q viene del decoder, **K y V vienen del encoder**: el decoder "le pregunta" al encoder qué entendió.

> Q/K/V es el **cómo** funciona la atención. Encoder / decoder / cross son el **dónde** se aplica.


## Ver attention en acción

📓 **Notebook opcional**: [`clase02/notebooks/02_attention_bertviz.ipynb`](notebooks/02_attention_bertviz.ipynb)

Usa BertViz para visualizar los pesos de atención de un modelo BERT real. Ideal para explorar después de clase.

> No lo ejecutamos en vivo (depende de bajar 680 MB del modelo). Está disponible para quien quiera meter mano.


# Bloque 3
## De base model a ChatGPT

---

*¿Cómo pasa un modelo de "completar texto" a "seguir instrucciones"?*


## El ciclo de vida de un LLM, de un vistazo

<img src="figures/lifecycle_overview.svg" alt="Tres etapas en línea: Pretraining produce un base model, SFT lo convierte en assistant, RLHF/DPO lo alinea con preferencias humanas" class="slide-figure" style="width: 92%;"/>

Vamos a recorrer cada etapa con ejemplos. La idea es que entiendan **qué cambia** entre una y otra: los datos, el objetivo, el costo, y qué modelo sale al final.


## Stage 1 — Qué te queda después del pretraining

<img src="figures/base_model_capabilities.svg" alt="Dos columnas: lo que un base model sí sabe (completar, imitar estilo, hechos del mundo) y lo que NO sabe (seguir instrucciones, detenerse, decir 'no sé', mantener un rol)" class="slide-figure" style="width: 68%;"/>

```
  Prompt:  "La capital de Francia es"

  base model →  "París, y también Roma es la capital de Italia,
                 mientras que Berlín lo es de Alemania. Las capitales
                 europeas suelen ser ciudades..."
```

Sabe la respuesta (París), pero **no se detiene**: sigue completando como si fuera un artículo de internet.


## Entrenar al assistant: cambiar el dataset

<img src="figures/training_assistant.svg" alt="Dataset de ~100K conversaciones USER/ASSISTANT escritas por humanos, usado para SFT" class="slide-figure" style="width: 88%;"/>

Misma arquitectura, mismo objetivo (predecir el próximo token). Pero los datos cambian: ahora son **conversaciones de alta calidad** escritas por personas siguiendo guías estrictas.

A esto se le llama **SFT** (Supervised Fine-Tuning).


## Antes vs después del fine-tuning

<img src="figures/after_finetuning.svg" alt="Comparativa: el base model genera más preguntas; el assistant detecta el bug y responde útil" class="slide-figure" style="width: 88%;"/>

El base model y el assistant son la **misma red neuronal**. Cambia solo en qué se la entrenó al final.


## Recolectar preferencias humanas

<img src="figures/comparisons_paperclips.svg" alt="Tres commits candidatos para el mismo diff; el humano elige el mejor" class="slide-figure" style="width: 78%;"/>

Para que el modelo se alinee con lo que **prefieren** las personas, primero hay que medir esa preferencia. La forma más escalable: pedirle al humano que **elija** entre respuestas, en vez de pedirle que **escriba** la perfecta.


## RLHF: el camino largo

<img src="figures/rlhf_pipeline.svg" alt="Tres pasos: 1) recolectar comparaciones, 2) entrenar un reward model que las imita, 3) usar PPO para que el LLM maximice ese reward" class="slide-figure" style="width: 86%;"/>

> 📖 *Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback (InstructGPT).*


## DPO: el atajo

<img src="figures/dpo_pipeline.svg" alt="DPO se saltea el reward model y el loop de PPO. Usa las comparaciones para optimizar el LLM directamente con una función de pérdida que sube la probabilidad del ganador y baja la del perdedor" class="slide-figure" style="width: 86%;"/>

> 📖 *Rafailov, R., et al. (2023). Direct Preference Optimization: Your Language Model is Secretly a Reward Model.*


## Base vs Instruct en la práctica

| Modelo | Variante | Uso típico |
|---|---|---|
| `Qwen3-8B` | base | fine-tuning propio, research |
| `Qwen3-8B-Instruct` | instruct | **el que usamos en este curso** |
| `Llama-3.1-8B` | base | continued pretraining |
| `Llama-3.1-8B-Instruct` | instruct | chatbots, asistentes |

> Cuando bajes un modelo de HuggingFace, fijate siempre el sufijo. Si te da respuestas raras o no responde, probablemente bajaste el base model.


# Bloque 4
## Scaling y estado del arte (2026)

---

*Por qué los modelos siguen creciendo, qué cambió en los últimos años, y qué hay en el mercado.*


## La era Chinchilla: "más N, más D, mejor modelo"

<img src="figures/scaling_laws.svg" alt="Curvas de loss vs tamaño del modelo, una por cada nivel de compute. Todas bajan, sin techo aparente. Pero la slide marca esto como histórico (~2022)" class="slide-figure" style="width: 80%;"/>

> 📖 *Hoffmann, J., et al. (2022). Training Compute-Optimal Large Language Models (Chinchilla).*


## Pero algo cambió: el data wall y los nuevos ejes

<img src="figures/data_wall.svg" alt="Izquierda: la oferta de texto público se aplana mientras la demanda de los LLMs sube; cruce ~2026-2028. Derecha: tres ejes nuevos donde escalar (post-training, test-time compute, synthetic data)" class="slide-figure" style="width: 86%;"/>


## Eje 1 — Post-training scaling

<img src="figures/post_training_scaling.svg" alt="Progresión SFT → RLHF/DPO → RLVR/GRPO, con un callout sobre DeepSeek-R1 como ejemplo concreto del cambio" class="slide-figure" style="width: 80%;"/>

**Las siglas:**
- **SFT** — Supervised Fine-Tuning. **RLHF** — RL from Human Feedback. **DPO** — Direct Preference Optimization. **RLVR** — RL with Verifiable Rewards. **GRPO** — Group Relative Policy Optimization (variante eficiente de PPO usada por DeepSeek-R1).

> 📖 *DeepSeek-AI (2025). DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.*


## Eje 2 — Test-time compute

<img src="figures/test_time_compute.svg" alt="Modelo clásico que responde directo (rápido, errores) vs modelo de razonamiento que muestra un bloque de thinking antes de responder (más lento, más correcto)" class="slide-figure" style="width: 86%;"/>

> 📖 *Snell, C., et al. (2024). Scaling LLM Test-Time Compute Optimally Can Be More Effective than Scaling Model Parameters.*


## El salto razonador vs clásico (2026)

<img src="figures/sparks_agi.svg" alt="Cuatro benchmarks (AIME, GPQA, SWE-Bench, Codeforces). En cada uno, el modelo razonador supera al clásico por 30-80 puntos porcentuales" class="slide-figure" style="width: 86%;"/>

> Misma generación, mismo año. La diferencia es cómo se entrenó **después** del pretraining.


## Ecosistema 2026 — un mapa rápido

| Categoría | Modelos relevantes |
|---|---|
| **Closed source** (top calidad) | GPT-5.4 (OpenAI), Claude Opus 4.6 (Anthropic), Gemini 3.1 Pro (Google), Grok 4 (xAI) |
| **Closed source** (balance) | Claude Sonnet 4.6, GPT-5.4-mini |
| **Open weight** (densos) | Llama 4 Scout (Meta), Qwen3 / Qwen3.5 (Alibaba), Mistral Small 4 |
| **Open weight** (MoE) | **Kimi K2.5** (Moonshot, 1T total / 32B activos), DeepSeek-R1 |
| **Razonadores** (open) | DeepSeek-R1, Qwen QwQ, Kimi K2.5 |
| **Para correr local** | Gemma 4, Qwen3-4B, Llama 3.2-3B (vía Ollama) |

**MoE (Mixture of Experts)**: solo se activan algunos "expertos" por request. Modelo grande, costo de modelo chico.


## ¿Dónde viven los modelos open weight?

<img src="figures/huggingface.svg" alt="Los tres pilares de HuggingFace: el Hub (repositorio de modelos y datasets), las librerías Python (transformers, datasets, trl, peft), y Spaces / Inference Endpoints" class="slide-figure" style="width: 88%;"/>


## Para este curso

Usamos **Qwen3 vía Groq** como modelo principal:

- **Gratis** con cuotas generosas para desarrollo.
- **Muy rápido** (Groq corre inferencia en hardware especializado).
- **Open weight** — podés bajarlo de HuggingFace y correrlo local también.

```bash
pip install groq
export GROQ_API_KEY="..."
```

Alternativa local opcional: **Ollama** con `qwen3:8b` o `qwen3:4b`.


# Bloque 5
## Prompting como ingeniería

---

*El prompt es la única interfaz. Usarla bien hace toda la diferencia.*


## Estructura de una llamada a un LLM

<img src="figures/prompt_structure.svg" alt="Tres bloques: SYSTEM (define rol/restricciones), USER (la pregunta), ASSISTANT (la respuesta)" class="slide-figure" style="width: 88%;"/>

Toda la "ingeniería de prompts" es jugar con estos tres campos.


## Output params: temperature, top-K, top-P

<img src="figures/sampling_params.svg" alt="Tres distribuciones de probabilidad mostrando cómo cambian con temperature baja/alta y con top-K/top-P" class="slide-figure" style="width: 88%;"/>

- **Temperature**: alta = creativo / diverso. Baja = predecible / factual.
- **Top-K**: corta a los K tokens más probables.
- **Top-P**: corta cuando la suma de probabilidades alcanza P.

> Bug clásico: con `temperature=0` o `top-K=1`, el modelo entra en **repetition loop**.


## Probá los parámetros vos mismo

📓 **Notebook**: [`clase02/notebooks/03_sampling_params.ipynb`](notebooks/03_sampling_params.ipynb)

Generá el mismo poema con `temperature=0.1` y `temperature=0.9`. Compará.


## System / Role / Contextual prompting

**System prompt** — define el rol y las restricciones del modelo.
```
Sos un ingeniero de software experto en Python.
Respondés conciso, técnico, solo en español.
```

**Role prompting** — asignás explícitamente un rol dentro del prompt.
```
Actuá como un profesor universitario que enseña con ejemplos prácticos.
Explicame qué es un decorador en Python.
```

**Contextual prompting** — proveés información que el modelo necesita para resolver.
```
Ley de Ohm: V = I × R
Pregunta: si I=2A y R=3Ω, ¿cuál es V?
```

> Los tres se pueden combinar.


## Zero / one / few-shot

<img src="figures/shot_spectrum.svg" alt="Tres tarjetas comparando zero-shot (sin ejemplos), one-shot (un ejemplo), few-shot (varios ejemplos)" class="slide-figure" style="width: 88%;"/>

> 📖 *Brown, T., et al. (2020). Language Models are Few-Shot Learners (GPT-3).*


## Probá zero-shot vs few-shot

📓 **Notebook**: [`clase02/notebooks/04_prompting_techniques.ipynb`](notebooks/04_prompting_techniques.ipynb)

Clasificá reseñas con un prompt zero-shot. Después con few-shot. Compará la calidad.


## Chain of Thought — "pensemos paso a paso"

```
PROBLEMA:
Cuando tenía 3 años, mi pareja tenía 3 veces mi edad.
Ahora tengo 20. ¿Cuántos años tiene mi pareja?

❌ Sin CoT (responde de una): "26"  (a veces se equivoca)

✓ Con CoT ("pensá paso a paso"):
  1. A los 3 años, mi pareja tenía 9.
  2. Diferencia de edad: 6 años.
  3. Hoy tengo 20 → mi pareja tiene 26.

  Respuesta: 26.
```

**Conexión con Karpathy**: el modelo necesita **tokens para pensar**. Si lo forzás a una respuesta corta, le sacás esa capacidad.

> 📖 *Wei, J., et al. (2022). Chain-of-Thought Prompting Elicits Reasoning in LLMs.*


## Structured output — JSON para integrar con código

```python
SYSTEM = '''
Analizás requerimientos. Respondés SOLO JSON con esta estructura:
{
  "tipo": "funcional" | "no_funcional" | "restriccion",
  "prioridad": "alta" | "media" | "baja",
  "componente": string,
  "resumen": string (max 15 palabras)
}
No incluyas nada más que el JSON.
'''

USER = "El sistema debe responder en <200ms al 95% de las requests."
```

```json
{"tipo": "no_funcional", "prioridad": "alta",
 "componente": "API", "resumen": "Latencia p95 menor a 200ms bajo carga normal"}
```

> Tip: usá `json-repair` para tolerar JSON parcialmente roto.


## Probá CoT y structured output

📓 **Notebook**: [`clase02/notebooks/05_cot_structured.ipynb`](notebooks/05_cot_structured.ipynb)

Resolvé un problema con y sin CoT. Después generá JSON parseable directamente desde el modelo.


## Prompt como código: variables y templating

<img src="figures/prompt_variables.svg" alt="Comparativa: a la izquierda, el prompt hardcodeado dentro de una función; a la derecha, un template separado con variables {{ review }} que el código solo renderea con Jinja" class="slide-figure" style="width: 86%;"/>

> 📖 *LangChain `PromptTemplate`, Jinja2, Mirascope, Promptfoo — todos los frameworks serios separan prompt de código.*


## Iteración: ¿cómo sé si mi prompt mejoró?

<img src="figures/prompt_evaluation.svg" alt="Tres pasos: armar dataset de tests, correr ambos prompts (v1 y v2), comparar accuracy. Abajo, una nota sobre cómo medir según la tarea (clasificación, generación, código, latencia)" class="slide-figure" style="width: 86%;"/>


## Práctica abierta — diseñá tu propio prompt

🌐 https://console.groq.com/playground · o desde el notebook 04.

**Tarea (15–20 min, en parejas):**

1. **Elegí** un caso de uso real para tu vida o tu trabajo: clasificar emails, extraer datos de un PDF, resumir reuniones, generar SQL desde lenguaje natural, etc.
2. **Armá 5 casos de prueba** con input + output esperado (esto es tu mini dataset).
3. **Escribí v1** del prompt: simple, zero-shot. Corré los 5 casos.
4. **Iterá a v2**: agregá rol + un par de ejemplos few-shot + restringí formato (JSON si aplica). Corré los 5 casos.
5. **Compará**: ¿cuántos casos pasan v1? ¿cuántos v2? Si empata, ¿cuál es más corto / barato?
6. **Reportá** el caso de uso, los dos prompts, y el accuracy de cada uno.

> El playground es para iterar rápido. La API + dataset de tests es para producción.


## Mejores prácticas (resumen)

- **Específico** > genérico. "Resumí en 3 bullets" > "resumí".
- **Ejemplos** > descripciones. Few-shot beat zero-shot cuando podés.
- **Instrucciones positivas** > restricciones. "Hacé X" > "no hagas Y".
- **Estructura clara**: usá tags (XML, JSON, markdown) para delimitar.
- **Iterá**: cambiá una cosa por vez. Documentá qué probaste.
- **Temperatura baja** para tareas factuales, alta para creatividad.
- **CoT** para razonamiento, **structured output** para integración con código.
- **Variables y templating** desde el día 1: el prompt es código, versionalo en git.
- **Iterá con dataset de tests** si la tarea importa. "Probé una vez y anduvo" no escala.


# Bloque 6
## Evals y monitoreo: del prompt al producto

---

*El prompt ya está bien diseñado. Ahora viene producción.*


## Evals offline vs online

<img src="figures/evals_offline_online.svg" alt="Dos paneles lado a lado: eval offline (antes del deploy, dataset estático, en CI) vs eval online (en producción, tráfico real, continuo)" class="slide-figure" style="width: 86%;"/>


## Qué loggear en producción

<img src="figures/monitoring_pipeline.svg" alt="Pipeline App → LLM → Logger middleware → Storage → Dashboard. Abajo, los 4 datos clave: input completo, output completo, latencia/costo, señal de calidad" class="slide-figure" style="width: 86%;"/>


## Drift: lo que cambia con el tiempo

<img src="figures/drift_detection.svg" alt="Cuatro tipos de drift: input drift, quality drift, cost drift, abuse drift. Abajo, un mini-dashboard mostrando accuracy bajando y costo subiendo a lo largo de 3 meses, con alerta disparada" class="slide-figure" style="width: 86%;"/>

> Tu sistema fue bueno el día del deploy. Tres meses después, no necesariamente.


## Herramientas (mayo 2026)

<img src="figures/tools_landscape.svg" alt="Tres columnas: evals offline (Promptfoo, Braintrust, DeepEval, OpenAI Evals, Inspect AI), evals de RAG (Ragas, preview Clase 3), monitoreo (Langfuse, LangSmith, Arize Phoenix, Helicone, TruLens). Una caja con 'por dónde empezar' según el caso" class="slide-figure" style="width: 86%;"/>


## El loop completo: prompt → producto → mejora

<img src="figures/feedback_loop.svg" alt="Ciclo de 6 nodos: diseño → eval offline → producción → eval online → análisis → feedback → vuelve al diseño. En el centro: 'PROMPT LIFECYCLE'" class="slide-figure" style="width: 86%;"/>

> Igual que el código, el prompt nunca está "terminado". Evoluciona con feedback de la realidad.


# Bloque 7
## Cierre y puente a Clase 3

---

*Lo que aprendimos hoy nos lleva directo al próximo problema.*


## Context window: el límite que motiva RAG

<img src="figures/context_window.svg" alt="Una barra apilada que muestra cómo SYSTEM + historial + documentos + USER compiten por el espacio del context window" class="slide-figure" style="width: 88%;"/>


## Las dos limitaciones que motivan Clase 3

Volvemos al hook del inicio. Vimos:

**1. El modelo alucina** (Bloque 1) — siempre genera el documento más plausible, exista o no. Sin grounding en datos reales, inventa con confianza.

**2. El context window es finito** (recién vimos) — y aunque sea de 10M tokens, **tu base de conocimiento no entra**.

```
                    ╔══════════════════════════════════╗
                    ║                                  ║
   Alucina ───────▶║   ¿Cómo le damos al modelo       ║
                    ║   conocimiento propio,           ║
   Context  ───────▶║   sin reentrenarlo?              ║
   limits           ║                                  ║
                    ╚══════════════╤═══════════════════╝
                                   │
                                   ▼
                              RAG (Clase 3)
```


## Lo que vimos hoy

| Bloque | Concepto clave |
|---|---|
| 1 — ¿Qué es un LLM? | Two files. Compresión de internet. Alucina por arquitectura. |
| 2 — Transformer | Attention en paralelo. Encoder vs decoder. Q/K/V. |
| 3 — Ciclo de vida | Pretrain → SFT → RLHF/DPO. Base vs Instruct. |
| 4 — Scaling 2026 | El pretraining se aplana; el eje hoy es post-training y test-time compute. |
| 5 — Prompting | System / few-shot / CoT / structured output / variables / iteración. |
| 6 — Evals y monitoreo | Eval offline + online. Logging. Drift. Loop de mejora continua. |

**Notebooks que quedaron para ejecutar:** `01_groq_intro`, `02_attention_bertviz`, `03_sampling_params`, `04_prompting_techniques`, `05_cot_structured`.


## Clase 3 — ¿qué viene?

Hoy aprendimos a **hablarle** a un LLM. En Clase 3 vemos cómo darle **conocimiento propio** sin reentrenarlo.

---

- **¿Por qué RAG?** — alucinación, knowledge cutoff, context limits.
- **Pipeline RAG naive** — chunking → embeddings → vector store → retrieval → augmentation.
- **Hybrid search** — BM25 (Clase 1) + búsqueda semántica (Clase 1) combinados.
- **RAG avanzado** — reranking, HyDE, parent-child chunks.
- **Práctica** — RAG funcional sobre documentos propios con ChromaDB + Ollama.


## 📚 Bibliografía

### Charla central
- **Karpathy, A. (2023). Intro to Large Language Models.**
  🔗 https://www.youtube.com/watch?v=zjkBMFhNj_g
  *La charla en la que se basa el grueso de los Bloques 1, 3, 4, 5.*

### Papers fundacionales
- Vaswani, A., et al. (2017). *Attention Is All You Need.* https://arxiv.org/abs/1706.03762
- Devlin, J., et al. (2018). *BERT.* https://arxiv.org/abs/1810.04805
- Brown, T., et al. (2020). *Language Models are Few-Shot Learners.* https://arxiv.org/abs/2005.14165
- Ouyang, L., et al. (2022). *Training language models to follow instructions with human feedback (InstructGPT).* https://arxiv.org/abs/2203.02155
- Wei, J., et al. (2022). *Chain-of-Thought Prompting.* https://arxiv.org/abs/2201.11903
- Hoffmann, J., et al. (2022). *Training Compute-Optimal LLMs (Chinchilla).* https://arxiv.org/abs/2203.15556
- Rafailov, R., et al. (2023). *Direct Preference Optimization.* https://arxiv.org/abs/2305.18290
- Berglund, L., et al. (2023). *The Reversal Curse.* https://arxiv.org/abs/2309.12288
- Hubinger, E., et al. (2024). *Sleeper Agents.* https://arxiv.org/abs/2401.05566

### Libros
- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). https://web.stanford.edu/~jurafsky/slp3/
- Tunstall, L., von Werra, L. & Wolf, T. (2022). *NLP with Transformers.* O'Reilly.
